# Intent Classification Evaluation & Latent Space Visualization

This notebook evaluates the performance and semantic representations of the **Llama-3.2-1B** model fine-tuned on the **Banking77** dataset. I compare the **Base Instruct Model** against our **Finetuned (DoRA) Model** to observe how the fine-tuning process has restructured the embedding space.

### Key Objectives:
1. **Embedding Extraction:** Extract high-dimensional intent representations from the last hidden layer of both models.
2. **Dimensionality Reduction:** Project these embeddings into 2D space using **t-SNE** and **UMAP** for visual inspection.
3. **Comparative Analysis:** Visualize how well the model clusters different intents before and after the fine-tuning process.
4. **Error Profiling:** Identify overlapping clusters that may lead to misclassifications in banking queries.

---
**Models being evaluated:**
* **Base:** `unsloth/Llama-3.2-1B-Instruct`
* **Finetuned:** `tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification`

**Dataset:**
* **Path:** `/kaggle/input/datasets/tranphungdinh/banking77-hf-split/split` (Test Split)

## 1. Install Dependencies

In [1]:
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade transformers trl liger-kernel datasets

# Install Visualization and ML tools (umap-learn is required for UMAP)
!pip install -q scikit-learn seaborn matplotlib pandas umap-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 10.0 MB/

## 2.Imports & Core Functions (Embeddings, t-SNE, UMAP)

In [2]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import warnings
from tqdm.auto import tqdm
from datasets import load_from_disk
from sklearn.manifold import TSNE
from unsloth import FastLanguageModel

# Suppress warnings for cleaner logs
warnings.filterwarnings("ignore")

# --- Constants & Setup ---
SYSTEM_PROMPT = "You are an expert banking intent classifier. Given a user query, output ONLY the exact intent name from the Banking77 dataset."
os.makedirs("outputs/embeddings", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)

# =========================================================
# 1. Embedding Extraction Functions
# =========================================================

def _build_embedding_prompt(text: str, tokenizer) -> str:
    """Build a prompt using only system and user messages to avoid assistant bias."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Classify the intent of this banking query: "{text}"'}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False, # We don't want the assistant header for embeddings
    )

def _mean_pool_with_mask(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Compute attention-mask-aware mean pooling."""
    mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.shape).float()
    sum_embeddings = (hidden_states * mask_expanded).sum(dim=1)
    sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
    return sum_embeddings / sum_mask

def extract_embeddings(model, tokenizer, texts, labels, batch_size=16, max_seq_length=512):
    """Extract mean-pooled hidden state embeddings."""
    model.eval()
    all_embeddings = []
    all_labels = []

    print(f"Extracting embeddings for {len(texts)} samples (batch_size={batch_size})...")

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting"):
        batch_texts = texts[i : i + batch_size]
        batch_labels = labels[i : i + batch_size]

        prompts = [_build_embedding_prompt(t, tokenizer) for t in batch_texts]
        
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_seq_length,
        ).to(model.device)

        # Forward pass to get hidden states
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # Extract the last hidden layer (-1)
        hidden_states = outputs.hidden_states[-1]

        # Apply attention-mask-aware mean pooling
        pooled = _mean_pool_with_mask(hidden_states, inputs["attention_mask"])

        all_embeddings.append(pooled.cpu().numpy())
        all_labels.extend(batch_labels)

    embeddings = np.concatenate(all_embeddings, axis=0)
    labels_array = np.array(all_labels)

    print(f"Extraction complete! Shape: {embeddings.shape}")
    return embeddings, labels_array

# =========================================================
# 2. t-SNE & UMAP Visualization Functions
# =========================================================

def visualize_tsne(embeddings, labels, model_name):
    """Apply t-SNE and generate a 2D scatter plot."""
    print(f"Running t-SNE for {model_name}...")
    tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, learning_rate=200.0, random_state=42, init="pca", metric="cosine")
    projections = tsne.fit_transform(embeddings)

    unique_labels = np.unique(labels)
    cmap = plt.cm.get_cmap("nipy_spectral", len(unique_labels))

    fig, ax = plt.subplots(figsize=(20, 16))
    for idx, label_name in enumerate(unique_labels):
        mask = labels == label_name
        ax.scatter(projections[mask, 0], projections[mask, 1], c=[cmap(idx)], s=15, alpha=0.7, label=label_name)

    ax.set_title(f"t-SNE Embedding Visualization - {model_name}", fontsize=16, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=6, ncol=2, markerscale=2)
    plt.tight_layout()

    save_path = f"outputs/plots/{model_name}_tsne.png"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved t-SNE plot to: {save_path}")

def visualize_umap(embeddings, labels, model_name):
    """Apply UMAP and generate a 2D scatter plot."""
    print(f"Running UMAP for {model_name}...")
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric="cosine", random_state=42)
    projections = reducer.fit_transform(embeddings)

    unique_labels = np.unique(labels)
    cmap = plt.cm.get_cmap("nipy_spectral", len(unique_labels))

    fig, ax = plt.subplots(figsize=(20, 16))
    for idx, label_name in enumerate(unique_labels):
        mask = labels == label_name
        ax.scatter(projections[mask, 0], projections[mask, 1], c=[cmap(idx)], s=15, alpha=0.7, label=label_name)

    ax.set_title(f"UMAP Embedding Visualization - {model_name}", fontsize=16, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=6, ncol=2, markerscale=2)
    plt.tight_layout()

    save_path = f"outputs/plots/{model_name}_umap.png"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved UMAP plot to: {save_path}")

2026-05-06 16:48:46.028563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778086126.415523      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778086126.523356      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778086127.497003      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778086127.497048      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778086127.497051      23 computation_placer.cc:177] computation placer alr

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. Load Dataset

In [3]:
print("Loading dataset from local path...")
DATASET_PATH = "/kaggle/input/datasets/tranphungdinh/banking77-hf-split/split"
dataset = load_from_disk(DATASET_PATH)

test_ds = dataset["test"]
texts = test_ds["text"]
true_labels = test_ds["label"] # Extracted as strings directly based on your dataset structure

print(f"Successfully loaded {len(texts)} test samples!")

Loading dataset from local path...
Successfully loaded 3080 test samples!


## 4. Process Base Model

In [4]:
BASE_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"
print(f"========== LOADING BASE MODEL: {BASE_MODEL_ID} ==========")

# Load model via Unsloth in 4-bit to save VRAM
model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

# Fix padding token if missing (required for batch processing)
if tokenizer_base.pad_token is None:
    tokenizer_base.pad_token = tokenizer_base.eos_token
tokenizer_base.padding_side = "right"

# 1. Extract
emb_base, lbl_base = extract_embeddings(model_base, tokenizer_base, texts, true_labels)
np.save("outputs/embeddings/BaseModel_embeddings.npy", emb_base)

# 2. Visualize
visualize_tsne(emb_base, lbl_base, "BaseModel")
visualize_umap(emb_base, lbl_base, "BaseModel")

# 3. Cleanup VRAM
del model_base
del tokenizer_base
torch.cuda.empty_cache()
gc.collect()
print("Base Model processing completed and VRAM cleared.")

========== LOADING BASE MODEL: unsloth/Llama-3.2-1B-Instruct ==========
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

[transformers] Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Extracting embeddings for 3080 samples (batch_size=16)...


Extracting:   0%|          | 0/193 [00:00<?, ?it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Extraction complete! Shape: (3080, 2048)
Running t-SNE for BaseModel...
Saved t-SNE plot to: outputs/plots/BaseModel_tsne.png
Running UMAP for BaseModel...
Saved UMAP plot to: outputs/plots/BaseModel_umap.png
Base Model processing completed and VRAM cleared.


## 5. Process Fintuned Model

In [5]:
FINETUNED_MODEL_ID = "tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification"
print(f"\n========== LOADING FINETUNED MODEL: {FINETUNED_MODEL_ID} ==========")

# Load finetuned model
model_ft, tokenizer_ft = FastLanguageModel.from_pretrained(
    model_name=FINETUNED_MODEL_ID,
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)

# Fix padding token if missing
if tokenizer_ft.pad_token is None:
    tokenizer_ft.pad_token = tokenizer_ft.eos_token
tokenizer_ft.padding_side = "right"

# 1. Extract
emb_ft, lbl_ft = extract_embeddings(model_ft, tokenizer_ft, texts, true_labels)
np.save("outputs/embeddings/FineTunedModel_embeddings.npy", emb_ft)

# 2. Visualize
visualize_tsne(emb_ft, lbl_ft, "FineTunedModel")
visualize_umap(emb_ft, lbl_ft, "FineTunedModel")

# 3. Cleanup VRAM
del model_ft
del tokenizer_ft
torch.cuda.empty_cache()
gc.collect()
print("Fine-tuned Model processing completed and VRAM cleared.")
print("All tasks finished successfully! Check the 'outputs/plots' folder for your visualizations.")


========== LOADING FINETUNED MODEL: tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification ==========
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

[transformers] Unsloth: Will load tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification as a legacy tokenizer.


Extracting embeddings for 3080 samples (batch_size=16)...


Extracting:   0%|          | 0/193 [00:00<?, ?it/s]

Extraction complete! Shape: (3080, 2048)
Running t-SNE for FineTunedModel...
Saved t-SNE plot to: outputs/plots/FineTunedModel_tsne.png
Running UMAP for FineTunedModel...
Saved UMAP plot to: outputs/plots/FineTunedModel_umap.png
Fine-tuned Model processing completed and VRAM cleared.
All tasks finished successfully! Check the 'outputs/plots' folder for your visualizations.
